# Week 03 - Transformer on Multi30k (German to English)

**Model file:** `week03_transformer.py` (`nn.Transformer` encoder-decoder with
sinusoidal positional encoding)
**Benchmark dataset:** Multi30k - the same 29,000 German/English sentence pairs
we used in week 2.

## Why the same dataset as week 2?

Because then the two models can be compared directly. Week 2 read the sentence
one word at a time with a GRU. The Transformer reads the whole sentence at once
with self-attention. Same data, same tokenizer, same metric - so any difference
in BLEU really comes from the model.

## What you will learn

1. What the padding mask and the causal (look-ahead) mask do.
2. Why a Transformer needs positional encoding.
3. How to train `nn.Transformer` with teacher forcing.
4. How to compare two model families on the same benchmark.

## Runtime

About 4-7 minutes on a GPU.

## 1. Setup

In [ ]:
import gzip
import importlib
import re
import shutil
import sys
import time
import urllib.request
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import common_eval
import week03_transformer as w3

importlib.reload(w3)
importlib.reload(common_eval)

print("model file loaded:", w3.__file__)

# PyTorch prints a warning because the causal mask is a float tensor while the
# padding mask is a bool tensor. Both are correct here, so we hide the message.
warnings.filterwarnings("ignore", message=".*mismatched key_padding_mask.*")
warnings.filterwarnings("ignore", message=".*nested tensors.*")

In [ ]:
# --- Notebook configuration -------------------------------------------------
SEED = 42
DATA_DIR = Path("./data/multi30k")

MIN_FREQ = 2
MAX_LEN = 30
BATCH_SIZE = 128
EPOCHS = 20
LR = 5e-4
LABEL_SMOOTHING = 0.1     # makes the model less over-confident, helps BLEU

D_MODEL = 256
N_HEAD = 8
N_LAYERS = 3
DIM_FF = 512
DROPOUT = 0.1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

w3.set_seed(SEED)
w3.DEVICE = DEVICE
print(f"device: {DEVICE}")

## 2. Data

Exactly the same loading and tokenizing code as week 2, so the two BLEU scores
can be compared.

In [ ]:
BASE_URL = "https://raw.githubusercontent.com/multi30k/dataset/master/data/task1/raw"
FILES = {
    "train": ["train.de", "train.en"],
    "valid": ["val.de", "val.en"],
    "test": ["test_2016_flickr.de", "test_2016_flickr.en"],
}


def download_multi30k(data_dir: Path):
    """Download and unzip the Multi30k text files if they are not there yet."""
    data_dir.mkdir(parents=True, exist_ok=True)
    for names in FILES.values():
        for name in names:
            target = data_dir / name
            if target.exists():
                continue
            url = f"{BASE_URL}/{name}.gz"
            gz_path = data_dir / f"{name}.gz"
            print("downloading", url)
            urllib.request.urlretrieve(url, gz_path)
            with gzip.open(gz_path, "rb") as source, open(target, "wb") as out:
                shutil.copyfileobj(source, out)
            gz_path.unlink()
    print("data ready in", data_dir.resolve())


download_multi30k(DATA_DIR)

TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", re.UNICODE)


def tokenize(text: str):
    """Lowercase and split into words and punctuation marks."""
    return TOKEN_PATTERN.findall(text.lower())


def read_split(name: str):
    de_file, en_file = FILES[name]
    de_lines = (DATA_DIR / de_file).read_text(encoding="utf-8").splitlines()
    en_lines = (DATA_DIR / en_file).read_text(encoding="utf-8").splitlines()
    pairs = []
    for de_line, en_line in zip(de_lines, en_lines):
        source, target = tokenize(de_line), tokenize(en_line)
        if 0 < len(source) <= MAX_LEN and 0 < len(target) <= MAX_LEN:
            pairs.append((source, target))
    return pairs


train_pairs = read_split("train")
valid_pairs = read_split("valid")
test_pairs = read_split("test")
print(f"train: {len(train_pairs)}   valid: {len(valid_pairs)}   test: {len(test_pairs)}")

In [ ]:
# Shared German + English vocabulary, same ids as the .py file expects.
PAD, BOS, EOS = w3.PAD_ID, w3.BOS_ID, w3.EOS_ID   # 0, 1, 2
UNK = 3

counter = Counter()
for source, target in train_pairs:
    counter.update(source)
    counter.update(target)

itos = ["<pad>", "<bos>", "<eos>", "<unk>"]
itos += [word for word, count in counter.most_common() if count >= MIN_FREQ]
stoi = {word: index for index, word in enumerate(itos)}
VOCAB_SIZE = len(itos)
print(f"vocabulary size: {VOCAB_SIZE}")


def encode(tokens):
    return [stoi.get(token, UNK) for token in tokens]


class TranslationDataset(Dataset):
    """Yields (source ids, target ids). The target keeps <bos> ... <eos>;
    the .py file shifts it into decoder input and decoder target itself."""

    def __init__(self, pairs):
        self.data = [(encode(s) + [EOS], [BOS] + encode(t) + [EOS])
                     for s, t in pairs]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]


def collate(batch):
    """Pad the batch to the longest sentence inside that batch."""
    source_max = max(len(s) for s, _ in batch)
    target_max = max(len(t) for _, t in batch)
    source = torch.full((len(batch), source_max), PAD, dtype=torch.long)
    target = torch.full((len(batch), target_max), PAD, dtype=torch.long)
    for row, (s, t) in enumerate(batch):
        source[row, :len(s)] = torch.tensor(s)
        target[row, :len(t)] = torch.tensor(t)
    return source, target


train_loader = DataLoader(TranslationDataset(train_pairs), batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate)
valid_loader = DataLoader(TranslationDataset(valid_pairs), batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate)
test_loader = DataLoader(TranslationDataset(test_pairs), batch_size=64,
                         shuffle=False, collate_fn=collate)

source, target = next(iter(train_loader))
print("batch shapes:", source.shape, target.shape)

## 3. The two masks

A Transformer looks at every position at the same time, so we must tell it
what it is **not** allowed to see.

* **Padding mask** - `True` where a position is only filler. Attention skips it.
* **Causal mask** - the decoder may only look at words it has already produced,
  never at future words. In PyTorch this is a matrix with `-inf` above the
  diagonal, added to the attention scores.

In [ ]:
example_target = target[:1, :-1]                      # decoder input of one sentence
causal = w3.causal_mask(example_target.size(1), torch.device("cpu"))
padding = w3.padding_mask(source[:1])

figure, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(causal.numpy(), cmap="gray_r")
axes[0].set_title("causal mask (white = allowed, black = -inf)")
axes[0].set_xlabel("key position")
axes[0].set_ylabel("query position")

axes[1].imshow(padding.numpy(), aspect="auto", cmap="gray_r")
axes[1].set_title("padding mask (black = real word, white = <pad>)")
axes[1].set_xlabel("source position")
axes[1].set_yticks([])
plt.tight_layout()
plt.show()

### Positional encoding

Self-attention has no idea about word order, so the model file adds a fixed
sine/cosine pattern to every embedding. Each position gets a unique pattern,
and nearby positions get similar patterns.

In [ ]:
positional = w3.PositionalEncoding(D_MODEL)
pattern = positional.pe[0, :60].numpy()

plt.figure(figsize=(9, 3.5))
plt.imshow(pattern.T, aspect="auto", cmap="RdBu")
plt.xlabel("position in the sentence")
plt.ylabel("embedding dimension")
plt.title("Sinusoidal positional encoding")
plt.colorbar()
plt.tight_layout()
plt.show()

## 4. Model

In [ ]:
model = w3.Seq2SeqTransformer(
    vocab=VOCAB_SIZE,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params / 1e6:.2f} M")
print(f"encoder layers: {N_LAYERS}   decoder layers: {N_LAYERS}   heads: {N_HEAD}")

## 5. Training

`w3.run_epoch` already does the teacher-forcing shift, builds both masks and
clips the gradients, so the training loop stays short.

We use **label smoothing 0.1**: instead of asking for probability 1.0 on the
correct word, we ask for 0.9 and spread 0.1 over the rest. This is what the
original Transformer paper does and it usually adds one or two BLEU points.

In [ ]:
train_loss_fn = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=LABEL_SMOOTHING)
plain_loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)   # used only for perplexity
optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.98), eps=1e-9)


@torch.no_grad()
def token_metrics(model, loader):
    """Cross entropy per token (no smoothing) and teacher-forced token accuracy."""
    model.eval()
    loss_sum, token_count, correct = 0.0, 0, 0
    for source, target in loader:
        source, target = source.to(DEVICE), target.to(DEVICE)
        target_in, target_out = target[:, :-1], target[:, 1:]
        mask = w3.causal_mask(target_in.size(1), DEVICE)
        logits = model(source, target_in, mask,
                       w3.padding_mask(source), w3.padding_mask(target_in))

        keep = target_out != PAD
        n_tokens = int(keep.sum())
        loss = plain_loss_fn(logits.reshape(-1, VOCAB_SIZE), target_out.reshape(-1))
        loss_sum += loss.item() * n_tokens
        token_count += n_tokens
        correct += int((logits.argmax(-1)[keep] == target_out[keep]).sum())
    return loss_sum / token_count, correct / token_count


history = {"train_loss": [], "valid_loss": [], "valid_acc": []}
best_valid_loss = float("inf")
best_state = None

start_time = time.time()
for epoch in range(1, EPOCHS + 1):
    train_loss = w3.run_epoch(model, train_loader, train_loss_fn, optimizer)
    valid_loss, valid_acc = token_metrics(model, valid_loader)

    history["train_loss"].append(train_loss)
    history["valid_loss"].append(valid_loss)
    history["valid_acc"].append(valid_acc)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        marker = "  <- best so far"
    else:
        marker = ""

    print(f"epoch {epoch:2d}/{EPOCHS}  train_loss={train_loss:.4f}  "
          f"valid_loss={valid_loss:.4f}  valid_ppl={np.exp(valid_loss):.2f}  "
          f"valid_token_acc={valid_acc:.4f}{marker}")

train_seconds = time.time() - start_time
print(f"\ntraining time: {train_seconds:.1f} s")
model.load_state_dict(best_state)
print(f"best validation loss: {best_valid_loss:.4f}")

In [ ]:
common_eval.plot_curves(history, ["train_loss", "valid_loss"],
                        title="Loss (train uses label smoothing, so it is higher)",
                        ylabel="loss")

## 6. Inference: greedy auto-regressive decoding

`w3.greedy_decode` starts from `<bos>` and adds the most likely next word until
every sentence has produced `<eos>`.

In [ ]:
def ids_to_words(ids):
    """Map ids to words. w3.strip_specials already removed <pad>/<bos>/<eos>."""
    return [itos[i] for i in ids]


def translate_loader(model, loader, max_len=MAX_LEN + 3):
    hypotheses, references = [], []
    for source, target in loader:
        predicted = w3.greedy_decode(model, source, max_len).cpu().tolist()
        for row in range(source.size(0)):
            hypotheses.append(ids_to_words(w3.strip_specials(predicted[row])))
            references.append(ids_to_words(w3.strip_specials(target[row].tolist())))
    return hypotheses, references


start_time = time.time()
test_hypotheses, test_references = translate_loader(model, test_loader)
decode_seconds = time.time() - start_time
print(f"decoded {len(test_hypotheses)} sentences in {decode_seconds:.1f} s")

for index in [0, 1, 5, 17, 42]:
    print(f"[{index}]")
    print("  source (de) :", " ".join(test_pairs[index][0]))
    print("  reference   :", " ".join(test_pairs[index][1]))
    print("  model       :", " ".join(test_hypotheses[index]))
    print()

## 7. Performance evaluation

In [ ]:
test_loss, test_token_acc = token_metrics(model, test_loader)
test_bleu = common_eval.corpus_bleu(test_hypotheses, test_references)

valid_hypotheses, valid_references = translate_loader(model, valid_loader)
valid_bleu = common_eval.corpus_bleu(valid_hypotheses, valid_references)

print("Multi30k results (German -> English)")
print(f"  validation BLEU-4        : {valid_bleu:.2f}")
print(f"  test BLEU-4              : {test_bleu:.2f}")
print(f"  test perplexity          : {np.exp(test_loss):.2f}")
print(f"  test token accuracy (TF) : {test_token_acc:.4f}")
print(f"  training time            : {train_seconds:.1f} s")

for order in [1, 2, 3, 4]:
    score = common_eval.corpus_bleu(test_hypotheses, test_references, max_n=order)
    print(f"  BLEU-{order}: {score:5.2f}")

hypothesis_length = sum(len(h) for h in test_hypotheses)
reference_length = sum(len(r) for r in test_references)
print(f"  length ratio (model / reference): {hypothesis_length / reference_length:.3f}")

### Compare with week 2

Fill in the numbers you got from the week 2 notebook and see the difference.
Both notebooks use the same data, the same tokenizer and the same BLEU code.

In [ ]:
WEEK02_TEST_BLEU = None      # <- paste your week 2 test BLEU here

print(f"{'model':<32}{'test BLEU-4':>12}")
print("-" * 44)
if WEEK02_TEST_BLEU is not None:
    print(f"{'week 2: GRU + attention':<32}{WEEK02_TEST_BLEU:>12.2f}")
else:
    print(f"{'week 2: GRU + attention':<32}{'(run week 2)':>12}")
print(f"{'week 3: Transformer':<32}{test_bleu:>12.2f}")

## 8. How long is the output allowed to be?

Greedy decoding stops at `<eos>`, but if the model never produces `<eos>` we
stop at `max_len`. Here we check how often that happens.

In [ ]:
too_long = sum(1 for h in test_hypotheses if len(h) >= MAX_LEN + 2)
print(f"sentences that hit the length limit: {too_long} / {len(test_hypotheses)}")

lengths_model = [len(h) for h in test_hypotheses]
lengths_reference = [len(r) for r in test_references]

plt.figure(figsize=(7, 3.5))
plt.hist(lengths_reference, bins=range(0, 35), alpha=0.6, label="reference")
plt.hist(lengths_model, bins=range(0, 35), alpha=0.6, label="model")
plt.xlabel("sentence length (words)")
plt.ylabel("count")
plt.legend()
plt.title("Output length vs. reference length")
plt.tight_layout()
plt.show()

## 9. Save the model

In [ ]:
save_path = Path("outputs_week03")
save_path.mkdir(exist_ok=True)
torch.save({"state_dict": model.state_dict(),
            "itos": itos,
            "config": {"d_model": D_MODEL, "nhead": N_HEAD, "layers": N_LAYERS,
                       "dim_ff": DIM_FF, "vocab_size": VOCAB_SIZE},
            "test_bleu": test_bleu},
           save_path / "transformer_multi30k.pt")
print(f"saved -> {(save_path / 'transformer_multi30k.pt').resolve()}")

## 10. Summary

| Item | Value |
|---|---|
| Dataset | Multi30k, German -> English (same as week 2) |
| Model | `nn.Transformer`, 3 encoder + 3 decoder layers, 8 heads |
| Main metric | corpus BLEU-4 on `test_2016_flickr` |

## Try it yourself

1. Set `N_LAYERS = 6` and `D_MODEL = 512`. Does BLEU improve enough to pay for
   the extra training time?
2. Turn label smoothing off (`LABEL_SMOOTHING = 0.0`). Compare BLEU and perplexity.
   Perplexity may look better while BLEU gets worse - why?
3. Add a learning-rate warm-up: start at 0 and grow linearly for the first 400
   steps, then decay with `1/sqrt(step)`. This is the schedule from the original paper.
4. Remove the positional encoding (return `x` unchanged in `PositionalEncoding.forward`).
   How far does BLEU drop? This shows how much word order matters.
5. Compare the decoding speed of week 2 and week 3. The Transformer trains in
   parallel but still decodes one word at a time - is it faster overall?